In [2]:
import sys
sys.path.append("../../..")
from utils.processing import json_file_access
from beir.datasets.data_loader import GenericDataLoader
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px

c:\Users\marvi\.conda\envs\masterthesis\Lib\site-packages\beir\datasets\data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
rq1_ir_res = json_file_access("../../../05_results/rq1/results_rq1.json", "r")
corpus, queries, qrels = GenericDataLoader(data_folder="../../../03_data/raw_data/beir_dbpedia/dbpedia-entity").load(split="test")

  0%|          | 0/4635922 [00:00<?, ?it/s]

In [4]:
from collections import defaultdict
results_rq1 = defaultdict(dict)

for key, value in rq1_ir_res.items():
    for query, result in value.items():
        results_rq1[key][query] = []
        for i, doc_id in enumerate(result.keys()):
            results_rq1[key][query].append(doc_id)

In [6]:
res_all = {}

configuration = {
    "cs_methods": ["LFMLocal", "GCE", "LocalTightnessExpansion"],
    "cs_core_node_number": [10,20,30]
}

# Iteratively add CS Top100
for cs in configuration["cs_methods"]:
    for cs_core_no in configuration["cs_core_node_number"]:
        res_all[cs + "_" + str(cs_core_no)] = json_file_access(f"../../../05_results/rq2/cs/kg1_results_rq2_cs_{cs}_{cs_core_no}.json","r")
# Manually add RQ1 results
res_all["RQ1" + "_" + str(0)] = results_rq1

In [ ]:
# Abbildung 28
cn = 10
cs_type = "GCE_" + str(cn)
ir_type = "hybrid"

overall = {"Relevante Dokumente CS": [], "Relevante Dokumente Initialmenge": []}
for query in res_all[cs_type][ir_type].keys():
    rel_docs_in_cs = 0
    rel_docs_in_ir = 0
    for doc, relevance in qrels[query].items(): # Loop through all qrel docs
        if relevance > 0:
            if doc in res_all[cs_type][ir_type][query]: # doc is in the results
                # Doc in CS Top100
                rel_docs_in_cs +=1
            if doc in res_all["RQ1_0"][ir_type][query][:cn]:
                # Doc in IR Top X
                rel_docs_in_ir +=1
    overall["Relevante Dokumente CS"].append(rel_docs_in_cs)
    overall["Relevante Dokumente Initialmenge"].append(rel_docs_in_ir)

df = pd.DataFrame(overall)
df = df.groupby(["Relevante Dokumente CS","Relevante Dokumente Initialmenge"]).size().reset_index().rename(columns={0:'count'})
df["count"] = df["count"]
fig = px.scatter(df, x="Relevante Dokumente Initialmenge", y="Relevante Dokumente CS",
	         size="count",color_discrete_sequence=['grey'], size_max=40, width=800, height=400)
fig.add_shape(type='line',
                x0=0,
                y0=0,
                x1=10,
                y1=10,
                line=dict(color='black',),
                xref='x',
                yref='y'
)
fig.update_layout({
"plot_bgcolor": "rgba(0, 0, 0, 0)",
"paper_bgcolor": "rgba(0, 0, 0, 0)",
"font_color":"black"
})
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

In [ ]:
# Abbildung 29
cn = 30
cs_type = "LFMLocal_" + str(cn)
ir_type = "boolean"

overall = {"Relevante Dokumente CS": [], "Relevante Dokumente Initialmenge": []}
for query in res_all[cs_type][ir_type].keys():
    rel_docs_in_cs = 0
    rel_docs_in_ir = 0
    for doc, relevance in qrels[query].items(): # Loop through all qrel docs
        if relevance > 0:
            if doc in res_all[cs_type][ir_type][query]: # doc is in the results
                # Doc in CS Top100
                rel_docs_in_cs +=1
            if doc in res_all["RQ1_0"][ir_type][query][:cn]:
                # Doc in IR Top X
                rel_docs_in_ir +=1
    overall["Relevante Dokumente CS"].append(rel_docs_in_cs)
    overall["Relevante Dokumente Initialmenge"].append(rel_docs_in_ir)

df = pd.DataFrame(overall)
df = df.groupby(["Relevante Dokumente CS","Relevante Dokumente Initialmenge"]).size().reset_index().rename(columns={0:'count'})
df["count"] = df["count"]
fig = px.scatter(df, x="Relevante Dokumente Initialmenge", y="Relevante Dokumente CS",
	         size="count",color_discrete_sequence=['grey'], size_max=40, width=800, height=400)
fig.add_shape(type='line',
                x0=0,
                y0=0,
                x1=25,
                y1=25,
                line=dict(color='black',),
                xref='x',
                yref='y'
                )
fig.update_layout({
"plot_bgcolor": "rgba(0, 0, 0, 0)",
"paper_bgcolor": "rgba(0, 0, 0, 0)",
"font_color":"black"
})
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()